# Energy Reconstruction Using CNN - Both Charges and Cos(Zenith)

In [1]:
import numpy as np
import tensorflow as tf
import os
import matplotlib.pyplot as plt

from datetime import datetime

from tensorflow import keras
from keras import layers
from keras import models
from keras import callbacks

#from tensorflow.keras.models import Sequential
#from tensorflow.keras.layers import Dense, Conv2D, Flatten
#from tensorflow.keras.callbacks import ModelCheckpoint

from data_tools import load_preprocessed, dataPrep, nameModel

simPrefix = os.getcwd()+'\\simdata'

In [2]:
tf.__version__

'2.6.0'

## Data Input

In [3]:
x, y = load_preprocessed(simPrefix, 'train')

Percentage of events with a NaN: 2.68


In [4]:
print(x.shape)
print(y.keys())
# each station has 2 tanks, each tank has 2 DOMs (high/log gain)
# each tank measures charge and time
# each station gives 2 charges and 2 times, 4 total pieces of data per station
# stations arranged in 10x10 square lattice, 2 corners of square unused
# charge measured in VEM, vertical equivalent muon

# 'dir' is true direction, rest of dir are reconstruted by simulations
# 'plane_dir' assumes shower is flat plane
# 'laputop_dir' performs likelihood analysis
# 'small_dir' compromises between plane and laputop

(549773, 10, 10, 4)
dict_keys(['comp', 'energy', 'dir', 'plane_dir', 'laputop_dir', 'small_dir'])


In [5]:
# 85/15 split for training/validation
energy = y['energy']
comp = y['comp']
theta, phi = y['dir'].transpose()
nevents = len(energy)
trainCut = (np.random.uniform(size=nevents) < 0.85)
testCut = np.logical_not(trainCut)

## Model Training

### Alpha Model
- Input: no charge merge, no time layers included, normalized data, combined with cosine of zenith angle
- Layers: Two convolutional layers for charge, then combined with zenith
- Output: Energy

In [6]:
# Name for model
key = 'q1q2cosZComp'
i = 0
while(os.path.exists('models/model_{}.h5'.format(key+str(i)))):
    i = i + 1
key = key+str(i)
numepochs = 10
# Data preparation: no merging of charge (q), no time layers included (t=False), data normalized from 0-1
prep = {'q':None, 't':False, 'normed':True, 'reco':'plane_'}

In [7]:
# Create model using functional API for multiple inputs
charge_input=keras.Input(shape=(10,10,2,))

conv1_layer = layers.Conv2D(64,kernel_size=3,padding='same',activation='relu')(charge_input)
#batch1_layer = layers.BatchNormalization()(conv1_layer) # default -> axis = -1, 
#drop1_layer = layers.Dropout(0.2)(batch1_layer)

#conv2_layer = layers.Conv2D(32,kernel_size=3,padding='same',activation='relu')(conv1_layer)
#batch2_layer = layers.BatchNormalization()(conv2_layer)
#drop2_layer = layers.Dropout(0.2)(batch2_layer)

conv3_layer = layers.Conv2D(16, kernel_size=3, padding='same',activation="relu")(conv1_layer)

flat_layer = layers.Flatten()(conv3_layer)
zenith_input=keras.Input(shape=(1,))
concat_layer = layers.Concatenate()([flat_layer,zenith_input])
#output = layers.Dense(1)(concat_layer)

dense1_layer = layers.Dense(128,activation='relu')(concat_layer)
#batch3_layer = layers.BatchNormalization()(dense1_layer)
#drop3_layer = layers.Dropout(0.2)(batch3_layer)

#dense2_layer = layers.Dense(128,activation='relu')(dense1_layer)
#batch4_layer = layers.BatchNormalization()(dense2_layer)
#drop4_layer = layers.Dropout(0.2)(batch4_layer)

dense3_layer = layers.Dense(128,activation="relu")(dense1_layer)

output = layers.Dense(1)(dense3_layer)

model = models.Model(inputs=[charge_input,zenith_input],outputs=output,name=key)

model.compile(loss='mean_squared_error', optimizer='adam', metrics=['mae','mse'])

## Old model used for reference
#model = Sequential(name=nameModel(prep, 'test'))  # Automatic naming for flexible assessment later
## Add model layers
#model.add(Conv2D(64, kernel_size=3, activation='relu', input_shape=(10,10,2)))
#model.add(Conv2D(32, kernel_size=3, activation='relu'))
#model.add(Flatten())
#model.add(Dense(1)) # No activation function for last layer of regression model

## Compile model
#model.compile(loss='mean_squared_error', optimizer='adam', metrics=['mae','mse'])

In [8]:
# Establish arrays to be trained on
x_i = dataPrep(x, y, **prep)
temp_y = energy

In [9]:
model.summary()

Model: "q1q2cosZComp0"
__________________________________________________________________________________________________
Layer (type)                    Output Shape         Param #     Connected to                     
input_1 (InputLayer)            [(None, 10, 10, 2)]  0                                            
__________________________________________________________________________________________________
conv2d (Conv2D)                 (None, 10, 10, 64)   1216        input_1[0][0]                    
__________________________________________________________________________________________________
conv2d_1 (Conv2D)               (None, 10, 10, 16)   9232        conv2d[0][0]                     
__________________________________________________________________________________________________
flatten (Flatten)               (None, 1600)         0           conv2d_1[0][0]                   
______________________________________________________________________________________

In [10]:
keras.utils.plot_model(model,"model.png")

('You must install pydot (`pip install pydot`) and install graphviz (see instructions at https://graphviz.gitlab.io/download/) ', 'for plot_model/model_to_dot to work.')


In [11]:
# Train
csv_logger = callbacks.CSVLogger('models/{}'.format(key))
early_stop = callbacks.EarlyStopping(patience=5, restore_best_weights=True) # default -> val_loss
checkpoint = callbacks.ModelCheckpoint('models/model_%s.h5' % key,save_best_only=True)
callbacklist = [early_stop, csv_logger,checkpoint]
history = model.fit(
    x=[x_i[0],x_i[1]], y=temp_y, epochs=numepochs,validation_split=0.15,callbacks=callbacklist)

Epoch 1/10
14354/14354 [==============================] - 146s 10ms/step - loss: 0.4948 - mae: 0.5740 - mse: 0.4948 - val_loss: 0.3810 - val_mae: 0.5262 - val_mse: 0.3810
Epoch 2/10
14354/14354 [==============================] - 148s 10ms/step - loss: 0.4487 - mae: 0.5627 - mse: 0.4487 - val_loss: 0.3947 - val_mae: 0.5321 - val_mse: 0.3947
Epoch 3/10
14354/14354 [==============================] - 153s 11ms/step - loss: 0.4452 - mae: 0.5609 - mse: 0.4452 - val_loss: 0.3950 - val_mae: 0.5324 - val_mse: 0.3950
Epoch 4/10
14354/14354 [==============================] - 148s 10ms/step - loss: 0.4439 - mae: 0.5603 - mse: 0.4439 - val_loss: 0.3796 - val_mae: 0.5251 - val_mse: 0.3796
Epoch 5/10
14354/14354 [==============================] - 147s 10ms/step - loss: 0.4423 - mae: 0.5595 - mse: 0.4423 - val_loss: 0.4458 - val_mae: 0.5611 - val_mse: 0.4458
Epoch 6/10
14354/14354 [==============================] - 149s 10ms/step - loss: 0.4413 - mae: 0.5591 - mse: 0.4413 - val_loss: 0.3825 - val_mae:

In [12]:
# Save model to file for easy loading
## WHERE ARE YOU SAVING TO?
np.save('models/model_%s.npy' % key,prep)
#model.save('models/model_%s.h5' % key)
f = open("results.txt", "a")
now = datetime.now()
f.write("{}\t{}\tepochs:{}\tloss:{},{}\n".format(
    now.strftime("%m/%d/%Y %H:%M:%S"),
    key,
    len(history.history['loss']),
    history.history['loss'][len(history.history['loss'])-1],
    history.history['val_loss'][len(history.history['loss'])-1]
))
f.close()